In [2]:
import itertools
import numpy as np
import cvxpy as cp

In [3]:
def minPIC_solver(U, H, min_rate):
    """
    Solves the minPIC optimization problem for a given U using a bracket + binary search
    on the imp_factor to find the smallest factor at which c[i] constraints become tight.

    :param U: Number of users (and receivers)
    :param H: Channel matrix of size (U, U)
    :param min_rate: Minimum required total data rates for each user (array of size U)
    :return: (best_sol, best_powers, best_data_rates)
    """

    permutations = list(itertools.permutations(range(U)))

    #-----------------------------------------------------------------------
    # Helper function: solve once for a given imp_factor
    # Returns:
    #   best_sol, best_powers, best_data_rates, constraints_tight_global
    #   (constraints_tight_global=True if there's a feasible solution with c[i] tight for all i)
    #-----------------------------------------------------------------------
    def solve_for_imp_factor(imp_factor):
        best_sol_local = float('inf')
        best_powers_local = None
        best_data_rates_local = None
        constraints_tight_global = False

        for pi in permutations:
            # Define optimization variables
            Rxx = {(i, j): cp.Variable(nonneg=True) for i in range(U) for j in range(U)}
            b   = {(i, j): cp.Variable() for i in range(U) for j in range(U)}
            c   = {i: cp.Variable() for i in range(U)}

            constraints = []
            for i in range(U):
                # S1, S2, S3
                S1 = [(i, j) for j in range(U)]
                S2 = [(j, i) for j in range(U) if j != i]
                S3 = [(j, k) for j in range(U) for k in range(U) if (j, k) not in (S1 + S2)]

                # Decoding order based on pi
                ordered_S1_S2 = sorted(S1 + S2, key=lambda x: (pi.index(x[0]), x[1]))
                decoding_order = list(ordered_S1_S2)

                # c[i] <= 0.5 * log_det(...) / log(2)
                H_unimportant = np.array([H[i][j] for (j, k) in S3]).reshape(1, -1)
                Rxx_unimportant = cp.diag(cp.hstack([Rxx[j, k] for (j, k) in S3]))
                constraints.append(
                    c[i] <= 0.5 *
                    cp.log_det(np.eye(H_unimportant.shape[0]) +
                               H_unimportant @ Rxx_unimportant @ H_unimportant.T
                              ) / np.log(2)
                )

                # c[i] >= 0
                constraints.append(c[i] >= 0)

                # Decoding-order constraints
                cumulative_sum = c[i]
                H_cum = [H[i][j] for (j, k) in S3]
                for idx, (j, k) in enumerate(decoding_order):
                    H_cum.append(H[i][j])
                    H_effective = np.array(H_cum).reshape(1, -1)
                    Rxx_effective = cp.diag(cp.hstack([Rxx[m, n]
                                                       for (m, n) in (S3 + decoding_order[:idx+1])]))
                    constraints.append(
                        cumulative_sum + b[j, k] <=
                        0.5 * cp.log_det(
                            np.eye(H_effective.shape[0]) +
                            H_effective @ Rxx_effective @ H_effective.T
                        ) / np.log(2)
                    )
                    cumulative_sum += b[j, k]

                # Sum of b[i, j] for all j = min_rate[i]
                # (In your comment you mention it "should be >=", so consider updating if needed)
                constraints.append(sum(b[i, j] for j in range(U)) == min_rate[i])

                # b[i,j] >= 0
                for j in range(U):
                    constraints.append(b[i, j] >= 0)

            # Objective
            # The user previously had:
            #   sum(Rxx) - imp_factor * sum(c[i])
            # We'll keep that logic, but we are searching for the minimal factor that makes c[i] tight.
            objective = cp.Minimize(
                sum(Rxx[i, j] for i in range(U) for j in range(U)) -
                imp_factor * sum(c[i] for i in range(U))
            )

            # Solve
            prob = cp.Problem(objective, constraints)
            try:
                prob.solve()
            except cp.SolverError:
                continue  # skip if solver fails

            if prob.status == cp.OPTIMAL and prob.value < best_sol_local:
                # Check if all c[i] constraints are tight
                # We do this by recomputing the right-hand side for c[i] and see if it's close
                all_tight = True
                for i in range(U):
                    S1 = [(i, j) for j in range(U)]
                    S2 = [(j, i) for j in range(U) if j != i]
                    S3 = [(j, k) for j in range(U) for k in range(U)
                          if (j, k) not in (S1 + S2)]
                    ordered_S1_S2 = sorted(S1 + S2, key=lambda x: (pi.index(x[0]), x[1]))
                    decoding_order = list(ordered_S1_S2)

                    H_unimportant = np.array([H[i][j] for (j, k) in S3]).reshape(1, -1)
                    Rxx_unimportant_val = np.diag(
                        np.hstack([Rxx[j, k].value for (j, k) in S3])
                    )

                    rhs_val = 0.5 * np.log2(
                        np.linalg.det(
                            np.eye(H_unimportant.shape[0]) +
                            H_unimportant @ Rxx_unimportant_val @ H_unimportant.T
                        )
                    )
                    lhs_val = c[i].value
                    # If difference bigger than threshold => not tight
                    if abs(lhs_val - rhs_val) > 1e-4:
                        all_tight = False
                        break

                if all_tight:
                    # Update best local solution
                    best_sol_local = prob.value
                    best_powers_local = {(i, j): Rxx[i, j].value for i in range(U) for j in range(U)}
                    best_data_rates_local = {i: sum(b[i, j].value for j in range(U)) for i in range(U)}
                    constraints_tight_global = True

        return best_sol_local, best_powers_local, best_data_rates_local, constraints_tight_global

    #-----------------------------------------------------------------------
    # 1) BRACKETING: find some highFactor where constraints become tight
    #-----------------------------------------------------------------------
    lowFactor = 1.0
    highFactor = lowFactor
    maxFactor = 1e6  # safeguard if we want to avoid infinite loops
    found_tight = False

    while highFactor <= maxFactor:
        best_sol, best_powers, best_data_rates, c_tight = solve_for_imp_factor(highFactor)
        if c_tight:
            # Found a factor that makes c[i] constraints tight
            found_tight = True
            break
        else:
            # Not tight, increase factor
            lowFactor = highFactor
            highFactor *= 2

    if not found_tight:
        # We never found a factor that makes c[i] constraints tight up to maxFactor
        print("No tight solution found up to imp_factor =", highFactor)
        return None, None, None

    #-----------------------------------------------------------------------
    # 2) BINARY SEARCH: find the minimal factor in [lowFactor, highFactor]
    #    for which constraints become tight
    #-----------------------------------------------------------------------
    best_sol_global = best_sol
    best_powers_global = best_powers
    best_data_rates_global = best_data_rates

    # We'll do ~20 iterations or until difference is tiny
    for _ in range(20):
        if (highFactor - lowFactor) < 1e-5:
            break
        midFactor = 0.5 * (lowFactor + highFactor)

        best_sol_mid, best_powers_mid, best_data_rates_mid, c_tight_mid = solve_for_imp_factor(midFactor)

        if c_tight_mid:
            # We can produce a tight solution at midFactor => search lower half
            highFactor = midFactor
            # Also check if it's better than the best solution we have so far
            if best_sol_mid < best_sol_global:
                best_sol_global = best_sol_mid
                best_powers_global = best_powers_mid
                best_data_rates_global = best_data_rates_mid
        else:
            # Not tight => we have to push factor up
            lowFactor = midFactor

    print("Binary search done. Found factor in [", lowFactor, ",", highFactor, "].")
    print("Best objective value:", best_sol_global)

    return best_sol_global, best_powers_global, best_data_rates_global


In [50]:
# Test case for 3 users
U = 3
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity
# put very small numbers for non diagonal elements
H[0, 1] = 0.00001
H[0, 2] = 0.00001
H[1, 0] = 0.00001
H[1, 2] = 0.00001
H[2, 0] = 0.00001
H[2, 1] = 0.00001


sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Trying imp_factor:  1
Trying permutation:  (0, 1, 2)
Receiver:  0
H_unimportant:  [[1.e-05 1.e-05 1.e-05 1.e-05]]
H_effective shape:  (1, 5)
H_effective shape:  (1, 6)
H_effective shape:  (1, 7)
H_effective shape:  (1, 8)
H_effective shape:  (1, 9)
Receiver:  1
H_unimportant:  [[1.e-05 1.e-05 1.e-05 1.e-05]]
H_effective shape:  (1, 5)
H_effective shape:  (1, 6)
H_effective shape:  (1, 7)
H_effective shape:  (1, 8)
H_effective shape:  (1, 9)
Receiver:  2
H_unimportant:  [[1.e-05 1.e-05 1.e-05 1.e-05]]
H_effective shape:  (1, 5)
H_effective shape:  (1, 6)
H_effective shape:  (1, 7)
H_effective shape:  (1, 8)
H_effective shape:  (1, 9)
LHS and RHS of c[i] constraint at receiver  0  :  8.104717923938096e-10 4.17685171540823e-11
LHS and RHS of c[i] constraint at receiver  1  :  3.9774712928175177e-10 1.2934923673004308e-10
LHS and RHS of c[i] constraint at receiver  2  :  1.5057537498974687e-10 1.4426951601138168e-10
Found a better solution with value:  3.000000003879455
Powers:  {(0, 0): 1

In [58]:
# Test case for 3 users
U = 3
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity
# put very small numbers for non diagonal elements
H[0, 1] = 0.00001
H[0, 2] = 0.00001
H[1, 0] = 0.00001
H[1, 2] = 0.00001
H[2, 0] = 0.00001
H[2, 1] = 0.00001


sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Binary search done. Found factor in [ 1.0 , 1.0 ].
Best objective value: 2.9999999978648186
Optimal Power Allocation: {(0, 0): 0.9999999936120395, (0, 1): 3.0688099621103955e-09, (0, 2): 3.937698473606117e-10, (1, 0): 0.7686700116777193, (1, 1): 0.23132999086858086, (1, 2): 2.0101045510915303e-09, (2, 0): 0.6533296867713515, (2, 1): 0.19770203421605687, (2, 2): 0.14896827721691994}
Optimal Data Rates: {0: 0.5, 1: 0.5, 2: 0.5}
Optimal cvx solution: 2.9999999978648186


In [6]:
# Test case for 3 users
U = 3
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity
# put very small numbers for non diagonal elements
H[0, 1] = 0.001
H[0, 2] = 0.001
H[1, 0] = 0.001
H[1, 2] = 0.001
H[2, 0] = 0.001
H[2, 1] = 0.001


sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Trying permutation:  (0, 1, 2)
Receiver:  0
H_unimportant:  [[0.001 0.001 0.001 0.001]]
H_effective shape:  (1, 5)
H_effective shape:  (1, 6)
H_effective shape:  (1, 7)
H_effective shape:  (1, 8)
H_effective shape:  (1, 9)
Receiver:  1
H_unimportant:  [[0.001 0.001 0.001 0.001]]
H_effective shape:  (1, 5)
H_effective shape:  (1, 6)
H_effective shape:  (1, 7)
H_effective shape:  (1, 8)
H_effective shape:  (1, 9)
Receiver:  2
H_unimportant:  [[0.001 0.001 0.001 0.001]]
H_effective shape:  (1, 5)
H_effective shape:  (1, 6)
H_effective shape:  (1, 7)
H_effective shape:  (1, 8)
H_effective shape:  (1, 9)
Found a better solution with value:  2.999570129340056
Powers:  {(0, 0): 1.0000029140130924, (0, 1): 2.523846598226756e-07, (0, 2): 1.385780397126835e-07, (1, 0): 0.0014145048827497624, (1, 1): 0.9985880895011363, (1, 2): 1.6247809294875464e-07, (2, 0): 0.001551166502775824, (2, 1): 0.00028114142877063916, (2, 2): 0.9981698948548685}
Data rates:  [array(0.5), array(5.01190542e-08), array(1.

In [59]:
# Test case for 3 users
U = 3
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity
# put very small numbers for non diagonal elements
H[0, 1] = 0.9
H[0, 2] = 0.001
H[1, 0] = 0.001
H[1, 2] = 0.001
H[2, 0] = 0.001
H[2, 1] = 0.001


sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Binary search done. Found factor in [ 5.016654968261719 , 5.01666259765625 ].
Best objective value: 0.06108674797918834
Optimal Power Allocation: {(0, 0): 2.582284988437918, (0, 1): 0.00021058721658485916, (0, 2): 3.0518347014489942e-09, (1, 0): 1.2712875851796119e-08, (1, 1): 1.9534878439605123, (1, 2): 0.00020924511305486407, (2, 0): 0.017379096726704162, (2, 1): 0.00021511235210015917, (2, 2): 0.9824103344909002}
Optimal Data Rates: {0: 0.49999999999999994, 1: 0.5000000000000001, 2: 0.5000000000000001}
Optimal cvx solution: 0.06108674797918834


In [57]:
# Test case for 2 users
U = 2
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity
# put very small numbers for non diagonal elements
H[0, 1] = 0.000001
H[1, 0] = 0.000001

sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Binary search done. Found factor in [ 1.0 , 1.0 ].
Best objective value: 1.9999999863725528
Optimal Power Allocation: {(0, 0): 0.9999999843411082, (0, 1): 4.740671876544762e-09, (1, 0): 0.75942040924968, (1, 1): 0.24057959111596466}
Optimal Data Rates: {0: 0.5, 1: 0.5}
Optimal cvx solution: 1.9999999863725528


In [56]:
# Test case for 2 users
U = 2
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity
# put very small numbers for non diagonal elements
H[0, 1] = 0.01
H[1, 0] = 0.01

sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Binary search done. Found factor in [ 1.0 , 1.0 ].
Best objective value: 1.9998000180553535
Optimal Power Allocation: {(0, 0): 0.9999000120365518, (0, 1): 1.3457862460262777e-09, (1, 0): 1.704277738958265e-05, (1, 1): 0.9998829644299729}
Optimal Data Rates: {0: 0.5, 1: 0.5}
Optimal cvx solution: 1.9998000180553535


In [55]:
# Test case for 2 users
U = 2
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity
# put very small numbers for non diagonal elements
H[0, 1] = 0.9
H[1, 0] = 0.00001

sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Binary search done. Found factor in [ 5.016853332519531 , 5.0168609619140625 ].
Best objective value: -0.9388697166681395
Optimal Power Allocation: {(0, 0): 2.3763312223504403, (0, 1): 0.2060672646141543, (1, 0): 1.1998898550830146e-07, (1, 1): 1.9535785757605477}
Optimal Data Rates: {0: 0.5, 1: 0.5}
Optimal cvx solution: -0.9388697166681395


In [4]:
# Test case for 2 users
U = 2
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity
# put very small numbers for non diagonal elements
H[0, 1] = 0.7
H[1, 0] = 0.5

sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Binary search done. Found factor in [ 3.4231948852539062 , 3.4232025146484375 ].
Best objective value: 0.5739103948456132
Optimal Power Allocation: {(0, 0): 1.6980056987577647, (0, 1): 8.678022052006848e-10, (1, 0): 2.4198673773085683e-09, (1, 1): 1.4245014216997975}
Optimal Data Rates: {0: 0.5, 1: 0.5}
Optimal cvx solution: 0.5739103948456132


In [12]:
# Test case for 2 users
U = 2
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity

sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Binary search done. Found factor in [ 1.0 , 1.0 ].
Best objective value: 1.9999999863725528
Optimal Power Allocation: {(0, 0): 0.9999999843411082, (0, 1): 4.740671876544762e-09, (1, 0): 0.75942040924968, (1, 1): 0.24057959111596466}
Optimal Data Rates: {0: 0.5, 1: 0.5}
Optimal cvx solution: 1.9999999863725528


In [9]:
# Test case for 2 users
U = 2
# H = np.random.rand(U, U) + 1j * np.random.rand(U, U)  # Random complex channel matrix
H = np.eye(U)  # Identity matrix for simplicity
# put very small numbers for non diagonal elements
H[0, 0] = 0.4
H[0, 1] = 0.01

H[1, 0] = 0.09


sigma2 = 1  # Noise PSD
# min_rate = np.array([0.5, 0.7, 0.6])  # Example minimum required data rates
min_rate = np.array([0.5, 0.5])  
# min_rate = np.array([2.0, 2.0, 2.0])
# min_rate = np.array([1.0, 1.0, 1.0])  

opt_sol, opt_powers, opt_data_rates = minPIC_solver(U, H, min_rate)
print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)

Binary search done. Found factor in [ 2.9109115600585938 , 2.910919189453125 ].
Best objective value: 7.157481735419421
Optimal Power Allocation: {(0, 0): 6.249342990719858, (0, 1): 1.2863397569502888e-08, (1, 0): 0.0001446669383638629, (1, 1): 1.050474849787235}
Optimal Data Rates: {0: 0.5, 1: 0.49999999999999994}
Optimal cvx solution: 7.157481735419421


In [5]:
U = 3
H = np.eye(U)
H[0,1] = 0.9
H[0,2] = 0.001
H[1,0] = 0.001
H[1,2] = 0.001
H[2,0] = 0.001
H[2,1] = 0.001

min_rate = [0.5, 0.5, 0.5]

[opt_sol, opt_powers, opt_data_rates] = minPIC_solver(U, H, min_rate);

print("Optimal Power Allocation:", opt_powers)
print("Optimal Data Rates:", opt_data_rates)
print("Optimal cvx solution:", opt_sol)


Binary search done. Found factor in [ 5.016654968261719 , 5.01666259765625 ].
Best objective value: 0.06108674797918834
Optimal Power Allocation: {(0, 0): 2.582284988437918, (0, 1): 0.00021058721658485916, (0, 2): 3.0518347014489942e-09, (1, 0): 1.2712875851796119e-08, (1, 1): 1.9534878439605123, (1, 2): 0.00020924511305486407, (2, 0): 0.017379096726704162, (2, 1): 0.00021511235210015917, (2, 2): 0.9824103344909002}
Optimal Data Rates: {0: 0.49999999999999994, 1: 0.5000000000000001, 2: 0.5000000000000001}
Optimal cvx solution: 0.06108674797918834
